## Python Questions

In [ ]:
import mysql.connector
import pandas as pd
import numpy as np

conn = mysql.connector.connect(host="localhost", user="root", password="Assasin#96", database="project_1")
cursor = conn.cursor()

def run_query(sql):
    cursor.execute(sql)
    return pd.DataFrame(cursor.fetchall(), columns=cursor.column_names)

movies         = run_query("SELECT * FROM movies")
genres         = run_query("SELECT * FROM genres")
movie_genres   = run_query("SELECT * FROM movie_genres")
cast           = run_query("SELECT * FROM cast")
crew           = run_query("SELECT * FROM crew")
movie_keywords = run_query("SELECT * FROM movie_keywords")

# adding year column
movies["release_date"] = pd.to_datetime(movies["release_date"])
movies["release_year"] = movies["release_date"].dt.year

#### Q1: List the total number of unique movies, genres, cast members and crew members collected across all tables.

In [46]:
pd.DataFrame({
    "Items":["unique Movies", "Unique Genres", "Unique Cast Members", "Unique crew Members"] ,
    "Counts": [movies['title'].nunique(), genres['genre_name'].nunique(), cast['actor_name'].nunique(), crew['person_name'].nunique()],
    "Values": [movies['title'].unique(), genres['genre_name'].unique(), cast['actor_name'].unique(), crew['person_name'].unique()]
})


,Items,Counts,Values
0,unique Movies,2452,"[Four Rooms, Star Wars, Finding Nemo, Forrest ..."
1,Unique Genres,19,"[Adventure, Fantasy, Animation, Drama, Horror,..."
2,Unique Cast Members,11291,"[Matthew McConaughey, Anne Hathaway, Michael C..."
3,Unique crew Members,2846,"[Jonathan Nolan, Christopher Nolan, Joss Whedo..."


* The dataset brings together 2,452 movies across 19 genres, featuring 11,291 cast members and 2,846 crew members

#### Q2: List all movies with missing or zero budget/revenue values, and decide how these should be treated for the rest of the analysis.

In [38]:
a2=pd.DataFrame({
    "Categories": ['Missing Budget', 'Missing Revenue', 'Zero Budget', 'Zero Revenue'],
    "Count": [movies['budget'].isna().sum(), movies['revenue'].isna().sum(), (movies['budget']==0).sum(), (movies['revenue']==0).sum()],
    "values": [movies.loc[movies['budget'].isna(), 'title'].tolist(), movies.loc[movies['revenue'].isna(),'title'].tolist(), 
               movies.loc[movies['budget']==0,'title'].tolist(), movies.loc[movies['revenue']==0,'title'].tolist()]
})

bad_data= movies[movies['budget'].isna() | movies['revenue'].isna() | (movies['budget']==0) | (movies['revenue']==0)]
print(a2)
print(f"\nMovies affected {len(bad_data)} out of {len(movies)}")

        Categories  Count                                             values
0   Missing Budget      0                                                 []
1  Missing Revenue      0                                                 []
2      Zero Budget     90  [The 400 Blows, 8½, Nosferatu, Persona, M, Def...
3     Zero Revenue    102  [The 400 Blows, 8½, Stalker, American Pie Pres...

Movies affected 133 out of 2503


* 133 of 2,503 movies (5.3%) have a zero budget, a zero revenue, or both: 90 zero budgets and 102 zero revenues, with 59 movies missing both. 
* As per the details given in the project document, "TMDB uses 0 to mean the figure is unknown"
* So we can treat this data as unknown and we can ignore it in analysis where budget and revenue calculation is needed or we can only consider movies where budget and revenue > 0

#### Q3: List the profit (revenue − budget) and return on investment for every movie that has valid budget and revenue figures.

In [ ]:
# In TMDB, 0 means "unknown", not "free" or "earned nothing".
# use only movies where BOTH budget and revenue are above 0. The originals stay untouched.

valid_movies = movies[(movies["budget"] > 0) & (movies["revenue"] > 0)]
valid_movies["profit"] = valid_movies["revenue"] - valid_movies["budget"]
valid_movies["roi_pct"] = (valid_movies["profit"] / valid_movies["budget"] * 100).round(1)   

valid_movies[["title", "budget", "revenue", "profit", "roi_pct"]].sort_values("profit", ascending=False)

,title,budget,revenue,profit,roi_pct
1136,Avatar,237000000,2923706026,2686706026,1133.6
1811,Avengers: Endgame,356000000,2799439100,2443439100,686.4
217,Titanic,200000000,2264162353,2064162353,1032.1
1436,Avatar: The Way of Water,460000000,2334484620,1874484620,407.5
1575,Star Wars: The Force Awakens,245000000,2068223624,1823223624,744.2
...,...,...,...,...,...
2213,Red Notice,160000000,178143,-159821857,-99.9
2412,The Killer,175000000,362113,-174637887,-99.8
2304,The Tomorrow War,200000000,14400000,-185600000,-92.8
2384,The Gray Man,200000000,454023,-199545977,-99.8


#### Q4: List the top 15 movies by return on investment, considering only movies with a budget of at least $1 million.

In [ ]:
valid_movies[valid_movies["budget"] >= 1_000_000].sort_values("roi_pct", ascending=False).head(15)[["title", "budget", "revenue", "roi_pct"]].reset_index()

,index,title,budget,revenue,roi_pct
0,2191,Dragon Ball Super: Broly,1000000,125002821,12400.3
1,152,Snow White and the Seven Dwarfs,1488423,184925486,12324.3
2,1211,The Rocky Horror Picture Show,1400000,171181400,12127.2
3,412,Rocky,1000000,117253345,11625.3
4,300,Gone with the Wind,4000000,402352579,9958.8
5,794,The Jungle Book,4000000,378000000,9350.0
6,1001,Cinderella,2900000,263600000,8989.7
7,78,Saw,1200000,104045735,8570.5
8,1055,One Hundred and One Dalmatians,3600000,303000000,8316.7
9,221,E.T. the Extra-Terrestrial,10500000,797307407,7493.4


#### Q5: List the number of movies released per year, and identify which year had the highest movie output in the dataset.

In [57]:
per_year = movies["release_year"].value_counts().sort_index(ascending=False)
print("Highest Movie output:", per_year.idxmax(), "with", per_year.max(), "movies")
per_year.to_frame("movies")

Highest Movie output: 2016 with 121 movies


,movies
release_year,
2026,8
2025,26
2024,43
2023,62
2022,65
...,...
1936,1
1931,2
1927,1


#### Q6: List all movies whose runtime is a statistical outlier compared to the rest of the dataset.

In [63]:
q1= movies['runtime'].quantile(0.25)
q3= movies['runtime'].quantile(0.75)
iqr=q3-q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr

print(f"Normal Movie runtime range: {low} to {high}")

outliers= movies[(movies['runtime'] < low) | (movies['runtime'] > high)]

print("Outliers:", len(outliers))
outliers[["title", "release_year", "runtime"]].sort_values("runtime", ascending=False).reset_index()


Normal Movie runtime range: 58.5 to 166.5
Outliers: 60


,index,title,release_year,runtime
0,2409,Zack Snyder's Justice League,2021,242
1,300,Gone with the Wind,1939,238
2,124,Once Upon a Time in America,1984,229
3,374,Lawrence of Arabia,1962,228
4,260,Ben-Hur,1959,212
5,2015,The Irishman,2019,209
6,136,Seven Samurai,1954,207
7,2150,Killers of the Flower Moon,2023,206
8,100,The Godfather Part II,1974,202
9,58,The Lord of the Rings: The Return of the King,2003,201


#### Q7: List each movie's primary genre (the first genre listed) and compare average popularity across primary genres.

In [69]:
# Primary genre = the first genre listed for each movie
primary = movie_genres.drop_duplicates(subset="movie_id", keep="first")
primary = primary.merge(genres, on="genre_id").merge(movies[["movie_id", "title", "popularity"]], on="movie_id")

primary[["title", "genre_name"]]                   

,title,genre_name
0,Interstellar,Adventure
1,Inception,Action
2,The Avengers,Science Fiction
3,The Dark Knight,Action
4,Avatar,Action
...,...,...
2494,[REC]²,Thriller
2495,Speak No Evil,Horror
2496,The Game Plan,Comedy
2497,Vantage Point,Drama


In [74]:
primary.groupby("genre_name")["popularity"].agg(movies="count", avg_popularity="mean").round(2).sort_values("avg_popularity", ascending=False)

,movies,avg_popularity
genre_name,,
Science Fiction,119,21.26
Music,10,18.32
Animation,139,16.81
Family,48,16.71
Action,474,16.39
War,23,15.47
Horror,223,14.85
Adventure,244,14.79
Fantasy,93,13.02


* Science fiction movies are the most popular ones

#### Q8: List the top 10 most prolific actors by number of movies, along with their average movie rating across those movies.

In [83]:
Actor_ratings= cast.merge(movies[['movie_id', 'vote_average']], on ='movie_id')
Actor_ratings = Actor_ratings.groupby('actor_name')['vote_average'].agg(movie_count ='count', avg_rating='mean').head(10).sort_values("movie_count",ascending=False)

print(f"10 most prolific actors by number of movies, along with their average movie rating: \n {Actor_ratings}")

10 most prolific actors by number of movies, along with their average movie rating: 
                     movie_count  avg_rating
actor_name                                 
50 Cent                       3       6.985
A.J. Cook                     2       6.700
A. Jay Radcliff               1       6.247
A. Russell Andrews            1       6.379
A.J. Benza                    1       6.886
A.J. Buckley                  1       6.748
A.J. Houghton                 1       6.758
AJ Bowen                      1       6.600
AJ Michalka                   1       6.781
Aamir Khan                    1       8.006


#### Q9: List any movie titles that appear more than once in the dataset, along with their release years, to check for duplicates or remakes.

In [92]:
dups = movies[movies.duplicated("title", keep=False)].sort_values(["title", "release_year"]).reset_index().head(200)
dups[["movie_id", "title", "release_year", "budget", "revenue"]]

,movie_id,title,release_year,budget,revenue
0,377,A Nightmare on Elm Street,1984,1800000,57004513
1,23437,A Nightmare on Elm Street,2010,35000000,115665418
2,812,Aladdin,1992,28000000,504050219
3,420817,Aladdin,2019,183000000,1054304000
4,12092,Alice in Wonderland,1951,3000000,57200000
...,...,...,...,...,...
95,60935,The Thing,2011,35000000,31505287
96,861,Total Recall,1990,65000000,261317921
97,64635,Total Recall,2012,125000000,198800000
98,1368168,Transformers,1987,110098767,274746695


#### Q10: List movies whose popularity score and vote count don't follow the general pattern between the two — i.e., unusually high popularity with a low vote count, or vice versa.

In [103]:
high_pop = movies["popularity"].quantile(0.75)
low_pop = movies["popularity"].quantile(0.25)

high_votes = movies["vote_count"].quantile(0.75)
low_votes = movies["vote_count"].quantile(0.25)

odd = movies[
    ((movies["popularity"] >= high_pop) & (movies["vote_count"] <= low_votes)) |
    ((movies["popularity"] <= low_pop) & (movies["vote_count"] >= high_votes))
].copy()

odd["issue"] = np.where(
    odd["popularity"] >= high_pop,
    "High popularity, low votes",
    "Low popularity, high votes"
)

odd[["title", "popularity", "vote_count", "issue"]].sort_values(
    "popularity", ascending=False
)

,title,popularity,vote_count,issue
2495,Disclosure Day,394.3350,2328,"High popularity, low votes"
2479,Backrooms,258.4740,2483,"High popularity, low votes"
2444,Mortal Kombat II,99.5231,2260,"High popularity, low votes"
2496,Lee Cronin's The Mummy,79.4601,2430,"High popularity, low votes"
2498,The Housemaid,41.4337,2789,"High popularity, low votes"
...,...,...,...,...
1317,X-Men: First Class,1.2614,13680,"Low popularity, high votes"
1209,X-Men: The Last Stand,1.1649,10511,"Low popularity, high votes"
1208,X2,0.8734,11013,"Low popularity, high votes"
130,The Lost World: Jurassic Park,0.8620,9356,"Low popularity, high votes"


In [104]:
odd['issue'].value_counts()

issue
High popularity, low votes    35
Low popularity, high votes    33
Name: count, dtype: int64

* we have 35 movies where popularity is too high for its votes
* we have 26 movies where votes are too high for its popularity
